In [ ]:
# parameters
# BINDER_FAST: set N=6, n_eps=8 for fast cloud execution
N = 8                # Hilbert space truncation
omega0_GHz = 5.0     # bare 0->1 frequency (GHz)
K_GHz = 0.2          # Kerr nonlinearity (GHz)
kappa_MHz = 1.0      # peak single-photon loss rate (MHz)
Gamma_MHz = 100.0    # bath FWHM (MHz)
nbar = 0.02          # thermal occupation
gamma_phi_MHz = 0.05 # pure dephasing rate (MHz)
epsilon_GHz = 0.1    # default drive amplitude for single-shot sims (GHz)
n_eps = 20           # number of points in the epsilon sweep


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.driven_kerr import (
    DrivenKerrConfig,
    make_H0,
    make_operators,
    make_jump_ops,
    J, J_vectorized,
    run_lindblad,
    run_redfield,
    check_positivity,
    effective_loss_rate,
    extract_excited_pop,
)

TWO_PI = 2 * np.pi


## Module 3b: Open-System Dynamics — Lindblad and Bloch–Redfield Methods

**Learning objectives:**
- Understand the Lindblad master equation and its jump operators
- Build physical intuition for the Lorentzian bath spectral density $J(\omega)$
- Run Methods A (Lindblad) and B (Bloch–Redfield) and compare their decay rates
- Identify the regime where Redfield produces unphysical negative populations
- Understand why neither Method A nor B can capture the drive-dependent decay rate change

---

**Sections:**
[1 The Structured Bath](#sec1) · 
[2 Method A (Lindblad)](#sec2) · 
[3 Method B (Bloch–Redfield)](#sec3) · 
[4 Rate vs. Drive Amplitude](#sec4) · 
[5 Exercises](#sec5)


<a id="sec1"></a>
## 1  The Structured Bath and the Spectral Density

### 1.1  Lindblad master equation

The standard open-system master equation (Lindblad form) for a system density matrix
$\rho$ coupled to a Markovian bath is:

$$\dot{\rho} = -i[\hat{H},\rho]
  + \sum_i \gamma_i\Bigl(\hat{L}_i\rho\hat{L}_i^\dagger
  - \frac{1}{2}\{\hat{L}_i^\dagger\hat{L}_i,\rho\}\Bigr).$$

The **jump operators** $\hat{L}_i$ encode dissipation channels; the rates $\gamma_i$ are
positive constants.  For our driven Kerr oscillator we use three channels:

| Channel | Jump operator | Physical process |
|---------|--------------|------------------|
| Loss (photon emission) | $\sqrt{\gamma_{\rm loss}}\hat{a}$ | Photon leaks to cold bath |
| Gain (thermal photon absorption) | $\sqrt{\gamma_{\rm gain}}\hat{a}^\dagger$ | Bath photon enters resonator |
| Pure dephasing | $\sqrt{\gamma_\phi}\hat{n}$ | Random phase kick, no energy exchange |

### 1.2  The Lorentzian spectral density

The rates $\gamma_{\rm loss}$ and $\gamma_{\rm gain}$ come from evaluating the
**two-sided spectral density** of the bath coupling $J(\omega)$ at the relevant
system frequencies.

For our Lorentzian bath centered at $\omega_f$ with peak value $\kappa$ and FWHM $\Gamma$:

$$J(\omega) =
\begin{cases}
\kappa\dfrac{(\Gamma/2)^2}{(\omega-\omega_f)^2+(\Gamma/2)^2}\times(1+\bar{n})
& \omega > 0 \;\text{(emission)}\\
\kappa\dfrac{(\Gamma/2)^2}{(-\omega-\omega_f)^2+(\Gamma/2)^2}\times\bar{n}
& \omega < 0 \;\text{(absorption)}
\end{cases}$$

where $\bar{n}$ is the bath thermal occupation.
The asymmetry between $\omega > 0$ and $\omega < 0$ enforces **detailed balance**:

$$\frac{J(-\omega)}{J(\omega)} = \frac{\bar{n}}{1+\bar{n}} = e^{-\hbar\omega/k_BT},$$

guaranteeing that the system thermalizes to the correct Boltzmann distribution at long times.

**Method A (Lindblad) rates** are fixed once:
$\gamma_{\rm loss} = J(+\omega_0)$ and $\gamma_{\rm gain} = J(-\omega_0)$.
They do **not** change as the drive amplitude $\epsilon$ is varied.

*Lab note: the Lorentzian bath models a structured electromagnetic environment, such as
a Purcell filter or a coupled transmission line with a finite bandwidth. Its FWHM $\Gamma$
determines the bandwidth of the environment. In the limit $\Gamma\to\infty$ (white bath)
the Lorentzian becomes flat and Method A becomes exact for any drive amplitude.*


In [ ]:
# Build the canonical configuration
cfg = DrivenKerrConfig(
    N        = N,
    omega0   = TWO_PI * omega0_GHz,
    K        = TWO_PI * K_GHz,
    omega_d  = TWO_PI * omega0_GHz * 1.001,  # 0.1% blue-detuned (5 MHz), avoids degeneracy
    omega_f  = TWO_PI * omega0_GHz,           # bath centered at omega0
    epsilon  = TWO_PI * epsilon_GHz,
    kappa    = TWO_PI * kappa_MHz * 1e-3,     # kappa in rad*GHz
    Gamma    = TWO_PI * Gamma_MHz * 1e-3,     # Gamma in rad*GHz
    nbar     = nbar,
    gamma_phi= TWO_PI * gamma_phi_MHz * 1e-3,
    k_max    = 5,
    n_t      = 512,
)

a_op, adag_op, n_op = make_operators(cfg.N)

print("Configuration summary:")
print(f"  ω₀/2π = {cfg.omega0/TWO_PI:.3f} GHz")
print(f"  K/2π  = {cfg.K/TWO_PI*1e3:.1f} MHz")
print(f"  κ/2π  = {cfg.kappa/TWO_PI*1e3:.2f} MHz  (bath peak rate)")
print(f"  Γ/2π  = {cfg.Gamma/TWO_PI*1e3:.1f} MHz  (bath FWHM)")
print(f"  n̄     = {cfg.nbar}")
print(f"  γ_φ/2π= {cfg.gamma_phi/TWO_PI*1e3:.3f} MHz")
print(f"  ε/2π  = {cfg.epsilon/TWO_PI*1e3:.1f} MHz  (={cfg.epsilon/cfg.K:.2f} K)")
print(f"  J(+ω₀) = {J(cfg.omega0, cfg)/TWO_PI*1e3:.4f} MHz  (= κ(1+n̄))")
print(f"  J(-ω₀) = {J(-cfg.omega0, cfg)/TWO_PI*1e3:.4f} MHz  (= κ·n̄)")


In [ ]:
# Plot J(omega) vs frequency
omega_arr = np.linspace(0.0, TWO_PI * 10.0, 1000)   # 0 to 10 GHz
J_arr = J_vectorized(omega_arr, cfg)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(omega_arr / TWO_PI, J_arr / (TWO_PI * 1e-3),
        lw=1.5, color="steelblue", label=r"$J(\omega)$ (positive $\omega$ side)")

# Annotate peak, FWHM, and relevant frequencies
ax.axvline(cfg.omega0 / TWO_PI, color="C2", ls="--", lw=1.2,
           label=rf"$\omega_0/2\pi = {omega0_GHz}$ GHz")
ax.axvline(cfg.omega_f / TWO_PI, color="C1", ls=":", lw=1.5,
           label=rf"$\omega_f/2\pi = {cfg.omega_f/TWO_PI:.3f}$ GHz (bath center)")

J_peak = J(cfg.omega_f, cfg)
ax.axhline(J_peak / (TWO_PI * 1e-3), color="gray", ls=":", lw=0.9, alpha=0.7)
ax.axhline(J_peak / (2 * TWO_PI * 1e-3), color="gray", ls="-.", lw=0.9, alpha=0.5,
           label=r"Half-power (FWHM = $\Gamma$)")

ax.annotate(
    rf"$\kappa(1+\bar{{n}})={J_peak/(TWO_PI*1e-3):.3f}$ MHz",
    xy=(cfg.omega_f / TWO_PI, J_peak / (TWO_PI * 1e-3)),
    xytext=(cfg.omega_f / TWO_PI + 0.3, J_peak / (TWO_PI * 1e-3) * 0.85),
    fontsize=9, color="C0",
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)

ax.set_xlabel(r"Frequency $\omega/2\pi$ (GHz)")
ax.set_ylabel(r"$J(\omega)/(2\pi)$ (MHz)")
ax.set_title("Bath spectral density (positive-frequency / emission side)")
ax.set_xlim(0, 10)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"J(ω₀) = {J(cfg.omega0, cfg)/TWO_PI*1e3:.4f} MHz   (≈ κ for ω₀ ≈ ω_f)")
print(f"J(ω₀ + Γ) = {J(cfg.omega0 + cfg.Gamma, cfg)/TWO_PI*1e3:.4f} MHz  (off-peak)")


<a id="sec2"></a>
## 2  Method A — Standard Lindblad

The Lindblad approach (Method A) is the simplest and fastest open-system simulation.
Jump operator rates are **frozen** at the bare-oscillator frequency $\omega_0$:

$$\gamma_{\rm loss} = J(+\omega_0), \qquad
\gamma_{\rm gain} = J(-\omega_0), \qquad
\gamma_{\rm deph} = \gamma_\phi.$$

These rates do not respond to the drive amplitude $\epsilon$.

We run `run_lindblad(cfg, rho0, tlist)` starting from $|1\rangle\langle 1|$
and track $P_1(t) = \langle 1|\rho(t)|1\rangle$.
The exponential decay rate $\Gamma_A$ is extracted by fitting
$P_1(t) \approx A e^{-\Gamma_A t} + C$.


In [ ]:
# Single run at a weak drive (default epsilon)
rho0 = qt.fock_dm(cfg.N, 1)   # start in |1>

# Time grid: simulate for ~5 lifetimes 1/kappa
T_sim = 5.0 / cfg.kappa        # seconds
tlist = np.linspace(0, T_sim, 300)

print(f"Simulation time = {T_sim*1e6:.1f} µs  (= 5 × 1/κ)")
print(f"Running Method A (Lindblad) at ε/2π = {cfg.epsilon/TWO_PI*1e3:.1f} MHz...")
result_A = run_lindblad(cfg, rho0, tlist)
print("Done.")

# Extract P1 and fit
pop1_A = extract_excited_pop(result_A, cfg.N)
rate_A = effective_loss_rate(result_A, tlist, cfg)
print(f"Fitted decay rate Γ_A/2π = {rate_A/TWO_PI*1e3:.4f} MHz")
print(f"Expected κ(1+2n̄)/2π    = {cfg.kappa*(1+2*cfg.nbar)/TWO_PI*1e3:.4f} MHz")


In [ ]:
# Plot P1(t) for Method A
from scipy.optimize import curve_fit

def exp_decay(t, A, gamma, C):
    return A * np.exp(-gamma * t) + C

popt, _ = curve_fit(exp_decay, tlist, pop1_A,
                    p0=[pop1_A[0] - pop1_A[-1], rate_A, pop1_A[-1]],
                    maxfev=5000)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Linear plot
ax = axes[0]
ax.plot(tlist * 1e6, pop1_A, lw=1.5, color="C0", label="Method A")
ax.plot(tlist * 1e6, exp_decay(tlist, *popt), "C1--", lw=1.2,
        label=rf"Fit: $\Gamma_A/2\pi = {rate_A/TWO_PI*1e3:.3f}$ MHz")
ax.set_xlabel(r"Time ($\mu$s)")
ax.set_ylabel(r"$P_{|1\rangle}(t)$")
ax.set_title("Method A: |1⟩ population decay (linear)")
ax.legend()
ax.grid(True, alpha=0.3)

# Semi-log plot
ax = axes[1]
pop_shifted = pop1_A - popt[2]  # subtract steady-state offset
fit_shifted = exp_decay(tlist, *popt) - popt[2]
pos = pop_shifted > 1e-4
ax.semilogy(tlist[pos] * 1e6, pop_shifted[pos], "C0-", lw=1.5, label="Method A (shifted)")
ax.semilogy(tlist[pos] * 1e6, fit_shifted[pos], "C1--", lw=1.2,
            label=rf"Exponential fit $\Gamma_A/2\pi = {rate_A/TWO_PI*1e3:.3f}$ MHz")
ax.set_xlabel(r"Time ($\mu$s)")
ax.set_ylabel(r"$P_{|1\rangle}(t) - P_\infty$")
ax.set_title("Method A: |1⟩ decay (semi-log)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print jump operators
c_ops = make_jump_ops(cfg)
print(f"\nMethod A jump operators ({len(c_ops)} total):")
for i, cop in enumerate(c_ops):
    rate_effective = (cop.dag() * cop).tr().real
    print(f"  L_{i}: norm² = {rate_effective:.4e}  (rate in rad·GHz)")


<a id="sec3"></a>
## 3  Method B — Bloch–Redfield

The Bloch–Redfield (BR) equation goes beyond Lindblad by keeping memory effects
and cross-relaxation terms.  The bath spectral density $J(\omega)$ is sampled not
just at the bare $\omega_0$ but at all transition frequencies including
**sidebands** $\omega_0 + k\omega_d$ (for integer $k$), weighted by
the appropriate matrix elements in the undriven Fock basis.

QuTiP implements BR via `brmesolve` with `a_ops = [[coupling_op, J(omega)], ...]`.

**Limitations of Method B:**
- Uses the undriven Fock basis — the drive enters only as a coherent perturbation,
  not as a change to the dissipation.
- At strong drive, the Fock-basis coupling amplitudes are redistributed among
  Floquet sidebands, but the bath remains sampled near $\omega_0 \pm k\omega_d$
  with bare Fock amplitudes. Since $J(\omega_0 \pm \omega_d) \approx 0$ for
  our narrow bath ($\Gamma / \omega_d \approx 0.02$), the sideband contributions
  are negligible and Method B behaves almost identically to Method A.
- At strong drive, Redfield can produce **small negative eigenvalues** in $\rho(t)$
  (unphysical!) because the Markov approximation breaks down in the driven basis.

*Lab note: for weak coupling to a flat (white) bath, Redfield and Lindblad are
equivalent. Redfield becomes important when the bath has structure on a scale
comparable to the system transition frequencies — here the Lorentzian bath has
$\Gamma = 100$ MHz while $\omega_0 = 5$ GHz, so the bath is structurally flat
at the scale of drive sidebands and Redfield adds little new information over Lindblad.
Method C (Floquet–Markov, Notebook 3c) properly accounts for the drive-dressed basis.*


In [ ]:
# Run Method B (Bloch-Redfield) at the same drive amplitude
print(f"Running Method B (Bloch-Redfield) at ε/2π = {cfg.epsilon/TWO_PI*1e3:.1f} MHz...")
result_B = run_redfield(cfg, rho0, tlist)
print("Done.")

pop1_B = extract_excited_pop(result_B, cfg.N)
rate_B = effective_loss_rate(result_B, tlist, cfg)
print(f"Fitted decay rate Γ_B/2π = {rate_B/TWO_PI*1e3:.4f} MHz")

# Check positivity
pos_check = check_positivity(result_B, tol=1e-4)
print(f"\nPositivity check (Method B):")
print(f"  Most negative eigenvalue: {pos_check['max_negative']:.2e}")
print(f"  Fraction of violated time steps: {pos_check['fraction_violated']:.3f}")
print(f"  Flagged as violation: {pos_check['flagged']}")


In [ ]:
# Overlay Methods A and B
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Time traces
ax = axes[0]
ax.plot(tlist * 1e6, pop1_A, lw=1.8, color="C0",
        label=rf"A (Lindblad) $\Gamma_A/2\pi={rate_A/TWO_PI*1e3:.3f}$ MHz")
ax.plot(tlist * 1e6, pop1_B, lw=1.5, color="C1", ls="--",
        label=rf"B (Redfield) $\Gamma_B/2\pi={rate_B/TWO_PI*1e3:.3f}$ MHz")
ax.set_xlabel(r"Time ($\mu$s)")
ax.set_ylabel(r"$P_{|1\rangle}(t)$")
ax.set_title(rf"Methods A vs B at $\varepsilon/2\pi = {cfg.epsilon/TWO_PI*1e3:.0f}$ MHz")
ax.legend()
ax.grid(True, alpha=0.3)

# Difference
ax = axes[1]
ax.plot(tlist * 1e6, np.abs(pop1_A - pop1_B), lw=1.2, color="C2")
ax.set_xlabel(r"Time ($\mu$s)")
ax.set_ylabel(r"$|P_A - P_B|$")
ax.set_title("Absolute difference |Method A − Method B|")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

rel_diff = abs(rate_A - rate_B) / rate_A
print(f"Relative rate difference |Γ_A - Γ_B| / Γ_A = {rel_diff:.3%}")
print("Methods A and B agree well here because Γ ≪ ω_d (narrow bath).")


<a id="sec4"></a>
## 4  Rate vs. Drive Amplitude

A key question for driven systems is: **does the decay rate change with drive amplitude?**

We sweep $\epsilon$ from $\sim 0.001K$ to $\sim 1.5K$ and extract the effective decay rate
for both Methods A and B.  We expect:

- **Method A:** flat, $\Gamma_A \approx \kappa(1 + 2\bar{n}) = $ constant, independent of $\epsilon$.
  The jump operators are frozen at $J(\omega_0)$ regardless of drive.
- **Method B:** nearly flat, slightly different at large $\epsilon$ due to
  Fock-basis redistribution, but since $J(\omega_0 \pm \omega_d) \approx 0$
  (bath narrow compared to sideband spacing), Method B also stays near $\kappa$.

**Foreshadowing:** In Notebook 3c we will see that Method C (Floquet–Markov)
predicts a rate that *falls by a factor of ~20* as $\epsilon \to K$.
This is entirely absent from Methods A and B — they lack the physics of
Floquet mode dressing.


In [ ]:
# Sweep epsilon from 0.001K to 1.5K
K_rad = cfg.K  # rad*GHz
eps_arr = np.geomspace(1e-3 * K_rad, 1.5 * K_rad, n_eps)

rates_A = np.full(n_eps, np.nan)
rates_B = np.full(n_eps, np.nan)

T_sim = 5.0 / cfg.kappa
tlist_sweep = np.linspace(0, T_sim, 200)

print(f"Sweeping ε from {eps_arr[0]/TWO_PI*1e3:.3f} to {eps_arr[-1]/TWO_PI*1e3:.1f} MHz")
print(f"({n_eps} points, {n_eps}×2 simulations — may take a few minutes)")

for i, eps in enumerate(eps_arr):
    cfg_i = cfg.replace(epsilon=eps)
    rho0_i = qt.fock_dm(cfg_i.N, 1)
    try:
        res_A = run_lindblad(cfg_i, rho0_i, tlist_sweep)
        rates_A[i] = effective_loss_rate(res_A, tlist_sweep, cfg_i)
    except Exception as e:
        print(f"  Method A failed at eps={eps/TWO_PI*1e3:.2f} MHz: {e}")
    try:
        res_B = run_redfield(cfg_i, rho0_i, tlist_sweep)
        rates_B[i] = effective_loss_rate(res_B, tlist_sweep, cfg_i)
    except Exception as e:
        print(f"  Method B failed at eps={eps/TWO_PI*1e3:.2f} MHz: {e}")
    if (i + 1) % 5 == 0 or i == n_eps - 1:
        print(f"  {i+1}/{n_eps} done")

print("Sweep complete.")


In [ ]:
# Plot rate vs epsilon / K
eps_over_K = eps_arr / cfg.K
kappa_MHz_val = cfg.kappa / TWO_PI * 1e3  # MHz

fig, ax = plt.subplots(figsize=(7, 4))

valid_A = np.isfinite(rates_A)
valid_B = np.isfinite(rates_B)

ax.semilogx(eps_over_K[valid_A], rates_A[valid_A] / TWO_PI * 1e3,
            "C0-o", ms=5, lw=1.5, label="A (Lindblad)")
ax.semilogx(eps_over_K[valid_B], rates_B[valid_B] / TWO_PI * 1e3,
            "C1-s", ms=5, lw=1.5, label="B (Bloch–Redfield)")
ax.axhline(kappa_MHz_val, color="gray", ls="--", lw=1,
           label=rf"$\kappa/2\pi = {kappa_MHz_val:.2f}$ MHz (bath peak)")
ax.axvline(1.0, color="k", ls=":", lw=0.8, alpha=0.6, label=r"$\varepsilon = K$")

ax.set_xlabel(r"Drive amplitude $\varepsilon/K$")
ax.set_ylabel(r"Decay rate $\Gamma/2\pi$ (MHz)")
ax.set_title("Methods A and B: rate vs. drive amplitude\n"
             "(Method C Floquet–Markov shown in Notebook 3c)")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
ax.set_ylim(bottom=0)

# Annotate flat behaviour
idx_mid = len(eps_over_K) // 2
ax.annotate("Both methods stay flat — drive does not\nchange Lindblad/Redfield rates",
            xy=(eps_over_K[idx_mid], rates_A[idx_mid] / TWO_PI * 1e3),
            xytext=(0.03, 0.5 * kappa_MHz_val),
            fontsize=9, color="C0",
            arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))

plt.tight_layout()
plt.show()

print(f"Rate variation across sweep:")
print(f"  Method A: min={rates_A[valid_A].min()/TWO_PI*1e3:.4f}  "
      f"max={rates_A[valid_A].max()/TWO_PI*1e3:.4f} MHz")
print(f"  Method B: min={rates_B[valid_B].min()/TWO_PI*1e3:.4f}  "
      f"max={rates_B[valid_B].max()/TWO_PI*1e3:.4f} MHz")
print("Both are essentially constant.  Method C will show a 20× drop at ε = K.")


**Key observation:** Both Method A and Method B give essentially flat rates across
the entire sweep from $\epsilon = 0.001K$ to $\epsilon = 1.5K$.
This is expected: both methods sample the bath only at the bare frequency $\omega_0$
(or at $\omega_0 \pm k\omega_d$ with the narrow bath giving near-zero weight there).

In Notebook 3c, Method C (Floquet–Markov) will show that the **physical decay rate**
drops by a factor of $\sim 20$ at $\epsilon = K$, because the dressed Floquet sideband
walks off the bath Lorentzian as $\epsilon$ increases.  Methods A and B are
**blind to this mechanism** — they are appropriate only when $\epsilon \ll K$.


<a id="sec5"></a>
## 5  Exercises

### Exercise 1: White bath limit

Repeat the rate sweep with a **white bath** by setting $\Gamma \to \infty$ (use
$\Gamma = 2\pi \times 50\,\text{GHz}$).  In this limit the Lorentzian spectral density
becomes flat and Method A (Lindblad) is the exact answer for any drive amplitude.

Does Method B converge to Method A in this limit?  Are the rates still flat?

**Hint:** use `cfg.replace(Gamma=TWO_PI * 50.0)` (50 GHz FWHM).


In [ ]:
# YOUR CODE HERE
# cfg_white = cfg.replace(Gamma=TWO_PI * 50.0, omega_f=cfg.omega0)
# Then repeat the sweep with cfg_white


### Exercise 2: Redfield positivity violation at strong drive

Run Method B at $\epsilon = 2K$ (well into the strong-drive regime) and check
whether `check_positivity` flags a violation.

1. Run `run_redfield(cfg_strong, rho0, tlist)`.
2. Call `check_positivity(result_B_strong, tol=1e-4)` and print the result.
3. Plot the time evolution of the minimum eigenvalue of $\rho(t)$.
4. Explain physically why Redfield can violate positivity at strong drive.

**Hint:** the minimum eigenvalue of $\rho$ must be $\geq 0$ for a valid quantum state.
At strong drive the Markov approximation (which assumes the bath correlation time is
shorter than the system evolution time) breaks down in the undriven basis, allowing
spurious negative populations.


In [ ]:
# YOUR CODE HERE
# cfg_strong = cfg.replace(epsilon=2 * cfg.K)
# result_B_strong = run_redfield(cfg_strong, rho0, tlist)
# pos = check_positivity(result_B_strong)
# print(pos)


In [ ]:
# Solution (hidden in PDF output)
cfg_strong = cfg.replace(epsilon=2 * cfg.K)
T_strong = 3.0 / cfg_strong.kappa
tlist_strong = np.linspace(0, T_strong, 200)
rho0_s = qt.fock_dm(cfg_strong.N, 1)

print("Running Method B at ε = 2K...")
result_B_strong = run_redfield(cfg_strong, rho0_s, tlist_strong)
pos = check_positivity(result_B_strong, tol=1e-4)
print(f"Positivity: max_negative = {pos['max_negative']:.2e}, flagged = {pos['flagged']}")

# Minimum eigenvalue over time
min_evals = [min(rho.eigenenergies()) for rho in result_B_strong.states]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(tlist_strong * 1e6, min_evals, lw=1.2, color="C3")
ax.axhline(0, color="gray", ls="--", lw=0.9)
ax.set_xlabel(r"Time ($\mu$s)")
ax.set_ylabel(r"$\lambda_{\min}(\rho)$")
ax.set_title(rf"Method B at $\varepsilon = 2K = {cfg_strong.epsilon/TWO_PI*1e3:.0f}$ MHz: minimum eigenvalue")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
